# Phase 3 — Hybrid retrieval and evidence coverage

This notebook implements **Phase 3**. Its historical filename includes Phase 4; B0 generation is now covered in the separate [Phase 4 notebook](phase_04_generator_and_b0_baseline.ipynb). This notebook itself does not run answer generation.

**What you will learn:** How to search passages from 26 complete annual reports, compare BM25 with embeddings, understand reciprocal rank fusion (RRF), and check whether selected evidence includes annotated source pages.

**Prerequisites:** Use the Financial RAG kernel and the existing Phase 1 and 2 data. The 13-page extraction audit was completed by an AI agent at the student's request on 22 October 2026; this is not human validation. These are development diagnostics with known parsing limitations.

**Execution boundaries:** Read only the ten development questions. Retrieval receives the question string, not reference answers, gold snippets or annotated page numbers. There are no Ollama requests, query rewrites, B1/B2 controller calls or held-out evaluations. The embedding model must still be loaded, but it does not generate answers.

Work through setup → index verification → one query's rankings → ten-question development diagnostics → six figures → exercises and report text. All new outputs go to `rag_study/outputs/retrieval_audit/phase3_line_aware_v2/`; `phase3/` contains historical results.

**Phase 2 revision:** this lesson uses line/word-safe chunks and the stricter table-admission checks. Quarantined table objects are excluded, but source-page text remains searchable. Later text chunks can still lack earlier year/unit headers; this lesson does not add headers, join pages or guarantee sufficient evidence. Earlier Phase 3 figures remain historical.

**Extraction-review update (22 October 2026):** at the student's request, a delegated AI agent reviewed all 13 source pages and current chunks. The worksheet records the review method explicitly. No student human review or generated-answer ratings are claimed. Inspect `agent_review_20261022/agent_review.md` under the Phase 2 output folder for the rubric and page findings.

In [ ]:
from pathlib import Path
import os, sys, json, time, hashlib
from datetime import datetime
from zoneinfo import ZoneInfo
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "rag_study/config.json").is_file())
assert Path(sys.prefix).resolve() == (ROOT / "rag_study/.venv").resolve(), "Select Financial RAG."
# Use cached models only: this lesson makes no downloads or network requests.
os.environ["HF_HOME"] = str(ROOT / "rag_study/hf_cache")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["ANONYMIZED_TELEMETRY"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
if str(ROOT / "rag_study") not in sys.path: sys.path.insert(0, str(ROOT / "rag_study"))
import study
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
OUT = ROOT / "rag_study/outputs/retrieval_audit/phase3_line_aware_v2"
OUT.mkdir(parents=True, exist_ok=True)
CONFIG = study.CONFIG
assert (CONFIG["retrieval_candidates"], CONFIG["rrf_constant"], CONFIG["top_k"], CONFIG["evidence_tokens"]) == (30, 60, 4, 2000)
tracked = ["config.json", "data/dev.jsonl", "data/chunks.jsonl", "data/manifest.csv", "data/index_signature.json", "data/chunking_manifest.json", "chunking.py", "table_quality.py", "study.py", "test_study.py"]
tracked.append("outputs/data_audit/phase2_line_aware_v2/development_document_audit.csv")
source_hashes = {name: study.digest(ROOT / "rag_study" / name) for name in tracked}
review = pd.read_csv(ROOT / "rag_study/outputs/data_audit/phase2_line_aware_v2/development_document_audit.csv")
pending = int(review.review_status.fillna("pending").eq("pending").sum())
print("Phase 2 extraction review rows pending:", pending)
print("Review method:", sorted(review.get("review_method", pd.Series(["unrecorded"])).dropna().unique()))
print("AI extraction review is not human answer scoring.")
print("Phase 3: cached retrieval only; no generator or held-out calls.")
display(pd.DataFrame({"Setting": ["Candidates per method", "RRF constant", "Whole chunks", "Generator evidence tokens"],
                      "Value": [30, 60, 4, 2000]}))

## Step 1 — Verify that the saved index is current

An index is like a library catalogue. A file's existence does not prove that its contents are correct. Check the chunks hash, embedding name and record count, then compare stored passages, source pages and chunk types in batches. This step does not call `build_index()` or rebuild the corpus.

The Qwen tokenizer only counts evidence tokens; it does not load Qwen's generation weights. The existing tokenizer may be in the project cache or user cache. The code searches offline and records the actual location.

In [ ]:
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from huggingface_hub.errors import LocalEntryNotFoundError
started = time.perf_counter()
retriever = study.Retriever()
signature = study.check_index_revision()
chunking_manifest = study.check_chunking_revision()
assert chunking_manifest["revision"] == "line_aware_v2"
assert signature["embedding"] == CONFIG["embedding"]
assert signature["count"] == retriever.collection.count() == len(retriever.chunks)
assert retriever.collection.metadata.get("hnsw:space") == "cosine"
assert len(retriever.by_id) == len(retriever.chunks)
for chunk in retriever.chunks:
    assert set(chunk) - {"chunking_revision"} == {"id", "doc_name", "page0", "pdf_page", "kind", "text", "embedding_tokens"}
    assert chunk["pdf_page"] == chunk["page0"] + 1
    assert chunk["chunking_revision"] == chunking_manifest["revision"]
    assert chunk["embedding_tokens"] <= CONFIG["chunk_tokens"]
seen = set()
for offset in range(0, signature["count"], 1000):
    batch = retriever.collection.get(limit=1000, offset=offset, include=["documents", "metadatas"])
    for cid, text, metadata in zip(batch["ids"], batch["documents"], batch["metadatas"]):
        original = retriever.by_id[cid]
        assert text == original["text"]
        assert all(metadata[k] == original[k] for k in ("doc_name", "page0", "pdf_page", "kind"))
        seen.add(cid)
assert seen == set(retriever.by_id)
cache_roots = [study.ROOT / "hf_cache/hub", Path.home() / ".cache/huggingface/hub"]
for cache in cache_roots:
    try:
        tokenizer_snapshot = Path(snapshot_download(CONFIG["generator_tokenizer"], cache_dir=cache, local_files_only=True))
        break
    except LocalEntryNotFoundError:
        continue
else:
    raise FileNotFoundError("Qwen tokenizer is not cached. Restore the Phase 0 tokenizer before continuing.")
tokenizer = AutoTokenizer.from_pretrained(tokenizer_snapshot, local_files_only=True)
embedding_snapshot = Path(snapshot_download(CONFIG["embedding"], local_files_only=True))
manifest = pd.read_csv(study.DATA / "manifest.csv")
assert len(manifest) == 26
print("Chunking revision:", chunking_manifest["revision"])
print("Indexed chunk types:", pd.Series(c["kind"] for c in retriever.chunks).value_counts().to_dict())
print("Index verified:", len(seen), "passages from", len(manifest), "reports")
print("Generator tokenizer:", type(tokenizer).__name__, "revision", tokenizer_snapshot.name)
print("Embedding revision:", embedding_snapshot.name)
print("Load/verification seconds:", round(time.perf_counter() - started, 2))

## Step 2 — Inspect one query: How does a question produce three rankings?

- **BM25** matches keywords and is useful for company names, years and financial terms. Its scores are not probabilities.
- **Dense retrieval** converts text into vectors and uses cosine distance to find passages with similar meanings. A smaller distance means greater similarity; it is not answer accuracy.
- **RRF** does not directly add raw scores from two different scales. A passage at rank r receives `1 / (60 + r)` from that ranking. Contributions are added if it appears in both rankings.

The function below reproduces the rankings for teaching, then checks that they agree with production `Retriever.retrieve()` output. Its only input is the question string.

In [ ]:
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
METHODS = ["BM25", "Dense", "Hybrid RRF"]
COLORS = {"BM25": "#527aa3", "Dense": "#bd8952", "Hybrid RRF": "#3a806b"}

def inspect_rankings(question_text):
    assert isinstance(question_text, str)
    scores = retriever.bm25.get_scores(study.tokens(question_text))
    lexical = [retriever.chunks[int(i)]["id"] for i in np.argsort(-scores, kind="stable")[:CONFIG["retrieval_candidates"]]]
    vector = retriever.model.encode([QUERY_PREFIX + question_text], normalize_embeddings=True)[0]
    response = retriever.collection.query(query_embeddings=[vector.tolist()], n_results=CONFIG["retrieval_candidates"], include=["distances"])
    dense = response["ids"][0]
    contributions = {}
    for method, ids in [("BM25", lexical), ("Dense", dense)]:
        for rank, cid in enumerate(ids, 1):
            contributions.setdefault(cid, {"BM25": 0.0, "Dense": 0.0})[method] = 1 / (CONFIG["rrf_constant"] + rank)
    order = sorted(contributions, key=lambda cid: (-sum(contributions[cid].values()), cid))[:CONFIG["retrieval_candidates"]]
    actual = retriever.retrieve(question_text)
    assert order == [c["id"] for c in actual], "Diagnostic and production rankings differ."
    assert all(np.isclose(c["retrieval_score"], sum(contributions[c["id"]].values())) for c in actual)
    return {"BM25": [retriever.by_id[cid] for cid in lexical],
            "Dense": [retriever.by_id[cid] for cid in dense], "Hybrid RRF": actual}, contributions

dev = study.read_jsonl(study.DATA / "dev.jsonl")
assert len(dev) == 10
question = dev[0]["question"]
example_rankings, example_contributions = inspect_rankings(question)
print("Development question:", question)
example_rows = []
for method, chunks in example_rankings.items():
    for rank, c in enumerate(chunks[:5], 1):
        example_rows.append({"method": method, "rank": rank, "chunk_id": c["id"], "report": c["doc_name"], "PDF page": c["pdf_page"], "kind": c["kind"]})
display(pd.DataFrame(example_rows))

## Step 3 — Enforce the evidence budget: Not all 30 candidates reach the model

Each method first retrieves 30 candidates, then applies the same rules to select at most 4 whole chunks within 2,000 **Qwen tokens**. Count citation labels, passage text and newlines between chunks. These counts differ from the BGE embedding-token lengths in Phase 2.

Phase 3 found that the old packer omitted separator newlines from its count. It now recounts the complete joined evidence. Two regression tests check that separators cannot push the context over budget and that the reported token count matches the actual text. Historical outputs have not been rewritten.

This step still does not answer the question or inspect its reference answer.

In [ ]:
contexts = {}
for method, candidates in example_rankings.items():
    text, chosen, reported_tokens = study.pack_context(candidates, tokenizer)
    actual_tokens = len(tokenizer.encode(text, add_special_tokens=False))
    assert reported_tokens == actual_tokens <= CONFIG["evidence_tokens"]
    assert len(chosen) <= CONFIG["top_k"]
    assert all(c["text"] in text for c in chosen)
    contexts[method] = (text, chosen, actual_tokens)
    print(method, ":", len(chosen), "whole chunks;", actual_tokens, "Qwen tokens")
print("\nHybrid final-context excerpt (not a generated answer):\n")
print(contexts["Hybrid RRF"][0][:1500])

In [ ]:
# 查看同一頁的所有 chunks，確認下一段是否保留後半句
for chunk in retriever.chunks:
    if chunk["doc_name"] == "PFIZER_2021_10K" and chunk["pdf_page"] == 19:
        print("\n---", chunk["id"], "---")
        print(chunk["text"])

## Step 4 — Ten development questions: Compare gold pages only after retrieval

Retrieve and pack evidence using the question first. **Only then** use development evidence-page annotations to calculate recall. Gold pages do not rank, filter or supplement the context.

`Evidence-page recall = number of annotated source pages retrieved / total annotated source pages for that question`. Compare `(doc_name, page0)` pairs, not page numbers alone.

BM25, Dense and Hybrid here are **retrieval diagnostics**, not B0/B1/B2 answer evaluations. Candidate recall examines the top 30 passages; packed recall examines passages that fit the evidence budget. Even recall = 1 does not establish that years, numbers and units are intact or that an answer is correct.

In [ ]:
metrics, candidate_rows, context_records, rank_curves = [], [], [], []
K_VALUES = [1, 2, 4, 8, 15, 30]
for number, q in enumerate(dev, 1):
    started = time.perf_counter()
    rankings, contributions = (example_rankings, example_contributions) if number == 1 else inspect_rankings(q["question"])
    # Gold annotations are consulted below only after retrieval and budget packing.
    packed = {method: study.pack_context(candidates, tokenizer) for method, candidates in rankings.items()}
    gold = {(e["doc_name"], int(e["evidence_page_num"])) for e in q["evidence"]}
    assert gold and all(any(c["doc_name"] == d and c["page0"] == p for c in retriever.chunks) for d, p in gold)
    for method, candidates in rankings.items():
        text, chosen, count = packed[method]
        assert count == len(tokenizer.encode(text, add_special_tokens=False)) <= 2000
        assert len(chosen) <= 4 and all(c["text"] in text for c in chosen)
        pages = lambda chunks: {(c["doc_name"], c["page0"]) for c in chunks}
        candidate_hits = len(gold & pages(candidates)); packed_hits = len(gold & pages(chosen))
        metrics.append({"financebench_id": q["financebench_id"], "company": q["company"], "method": method,
                        "gold_pages": len(gold), "candidate_hits": candidate_hits, "packed_hits": packed_hits,
                        "candidate_page_recall": candidate_hits / len(gold), "packed_page_recall": packed_hits / len(gold),
                        "packed_all_gold_pages": packed_hits == len(gold), "chunks": len(chosen), "evidence_tokens": count,
                        "packed_unique_pages": len(pages(chosen))})
        chosen_ids = {c["id"] for c in chosen}
        for rank, c in enumerate(candidates, 1):
            candidate_rows.append({"financebench_id": q["financebench_id"], "method": method, "rank": rank,
                                   "chunk_id": c["id"], "doc_name": c["doc_name"], "page0": c["page0"],
                                   "pdf_page": c["pdf_page"], "kind": c["kind"], "selected": c["id"] in chosen_ids,
                                   "on_gold_page": (c["doc_name"], c["page0"]) in gold})
        context_records.append({"financebench_id": q["financebench_id"], "method": method,
                                "query": q["question"], "chunk_ids": [c["id"] for c in chosen],
                                "evidence_tokens": count, "context": text})
        for k in K_VALUES:
            rank_curves.append({"financebench_id": q["financebench_id"], "method": method, "k": k,
                                "page_recall": len(gold & pages(candidates[:k])) / len(gold)})
    print(f"{number:02d}/10 {q['financebench_id']} checked")
metrics = pd.DataFrame(metrics); candidate_rows = pd.DataFrame(candidate_rows); rank_curves = pd.DataFrame(rank_curves)
summary = metrics.groupby("method", sort=False).agg(questions=("financebench_id", "count"),
    mean_candidate_page_recall=("candidate_page_recall", "mean"), mean_packed_page_recall=("packed_page_recall", "mean"),
    questions_with_all_gold_pages=("packed_all_gold_pages", "sum"), mean_evidence_tokens=("evidence_tokens", "mean"))
assert len(metrics) == 30 and len(candidate_rows) == 900
metrics.to_csv(OUT / "development_retrieval_metrics.csv", index=False)
candidate_rows.to_csv(OUT / "candidate_rankings.csv", index=False)
rank_curves.to_csv(OUT / "recall_by_rank.csv", index=False)
summary.to_csv(OUT / "retrieval_summary.csv")
study.write_jsonl(OUT / "packed_contexts.jsonl", context_records)
display(summary.round(3))

## Step 5 — Figures 1–3: Read recall results, then inspect the budget

**P3_1:** Packed evidence-page recall for each question. A dark cell means all annotated pages were retrieved; it does not mean the answer is correct.

**P3_2:** The change in mean recall from 30 candidates to the budget-packed context. A decrease means that an evidence page appeared among the candidates but was not retained.

**P3_3:** Actual Qwen evidence tokens per question. The red line marks the 2,000-token limit. Using more tokens is not necessarily better.

In [ ]:
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
order = [q["financebench_id"] for q in dev]
labels = [q["financebench_id"].removeprefix("financebench_id_") + " / " + q["company"] for q in dev]
heat = metrics.pivot(index="financebench_id", columns="method", values="packed_page_recall").loc[order, METHODS]
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(heat.values, cmap="YlGnBu", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(3), METHODS); ax.set_yticks(range(10), labels)
for y in range(10):
    for x in range(3): ax.text(x, y, f"{heat.iloc[y, x]:.2f}", ha="center", va="center", color="white" if heat.iloc[y, x] > .6 else "black")
ax.set_title("Development evidence-page recall in packed context (not answer accuracy)")
fig.colorbar(im, ax=ax, label="Gold-page recall"); fig.tight_layout(); fig.savefig(OUT / "P3_1_packed_page_recall.png", dpi=180); plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(3)
for shift, col, label, color in [(-.18, "candidate_page_recall", "30 candidates", "#a8b9c8"), (.18, "packed_page_recall", "Budget-packed: at most 4 chunks", "#3a806b")]:
    values = metrics.groupby("method")[col].mean().reindex(METHODS)
    bars = ax.bar(x + shift, values, width=.35, label=label, color=color)
    ax.bar_label(bars, fmt="%.2f", padding=3)
ax.set_xticks(x, METHODS); ax.set_ylim(0, 1.18); ax.set_ylabel("Mean gold-page recall over 10 questions")
ax.set_title("Candidate coverage versus evidence retained for the generator"); ax.legend(fontsize=9)
fig.tight_layout(); fig.savefig(OUT / "P3_2_candidate_vs_packed.png", dpi=180); plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
for i, method in enumerate(METHODS):
    values = metrics.query("method == @method").set_index("financebench_id").loc[order, "evidence_tokens"]
    ax.bar(np.arange(10) + (i-1)*.25, values, width=.24, label=method, color=COLORS[method])
ax.axhline(2000, color="#b34242", linestyle="--", label="2,000-token limit")
ax.set_xticks(range(10), [q["financebench_id"][-5:] for q in dev]); ax.set_ylim(0, 2250)
ax.set_xlabel("Development question ID"); ax.set_ylabel("Actual Qwen evidence tokens")
ax.set_title("Whole-chunk packing includes source labels and separators"); ax.legend(ncol=2, fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "P3_3_evidence_token_budget.png", dpi=180); plt.show()

## Step 6 — Figures 4–6: Why might the top RRF result be unhelpful?

**P3_4:** BM25 and Dense contributions to the first question's RRF scores. The two colours add to the total score; this is not confidence.

**P3_5:** Annotated-page recall as the number of retained candidates k increases. These candidates **have not been filtered by the token budget**, so recall at k=4 need not equal packed-context recall.

**P3_6:** Four chunks may come from repeated source pages. This plot shows the number of unique source pages in each question's final context. Four passages do not necessarily mean four pages.

In [ ]:
example_ids = [c["id"] for c in example_rankings["Hybrid RRF"][:12]]
trace = pd.DataFrame([{"chunk_id": cid, **example_contributions[cid],
                       "doc_name": retriever.by_id[cid]["doc_name"], "pdf_page": retriever.by_id[cid]["pdf_page"]}
                      for cid in example_ids])
trace.to_csv(OUT / "example_rrf_contributions.csv", index=False)
fig, ax = plt.subplots(figsize=(10, 6))
y = np.arange(len(trace))
ax.barh(y, trace["BM25"], label="BM25 rank contribution", color=COLORS["BM25"])
ax.barh(y, trace["Dense"], left=trace["BM25"], label="Dense rank contribution", color=COLORS["Dense"])
ax.set_yticks(y, [f"{row.doc_name} / p{row.pdf_page}" for row in trace.itertuples()]); ax.invert_yaxis()
ax.set_xlabel("RRF score: sum of 1 / (60 + rank)"); ax.set_title("First development query: why these chunks rank highly")
ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(OUT / "P3_4_rrf_contributions.png", dpi=180); plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
for method in METHODS:
    values = rank_curves.query("method == @method").groupby("k").page_recall.mean().reindex(K_VALUES)
    ax.plot(K_VALUES, values, marker="o", label=method, color=COLORS[method])
ax.set_xticks(K_VALUES); ax.set_ylim(-.03, 1.05); ax.set_xlabel("Top-k candidates before token-budget packing")
ax.set_ylabel("Mean annotated-page recall"); ax.set_title("Development candidate recall by rank cutoff")
ax.legend(); ax.grid(alpha=.2); fig.tight_layout(); fig.savefig(OUT / "P3_5_recall_by_rank.png", dpi=180); plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
for i, method in enumerate(METHODS):
    values = metrics.query("method == @method").set_index("financebench_id").loc[order, "packed_unique_pages"]
    ax.bar(np.arange(10)+(i-1)*.25, values, width=.24, label=method, color=COLORS[method])
ax.set_xticks(range(10), [q["financebench_id"][-5:] for q in dev]); ax.set_yticks(range(5)); ax.set_ylim(0, 4.8)
ax.set_xlabel("Development question ID"); ax.set_ylabel("Distinct (report, page) pairs")
ax.set_title("Unique source pages in the packed context (not gold-page count)")
ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(OUT / "P3_6_unique_pages.png", dpi=180); plt.show()

### Observations from the current run

The following table is calculated from this execution rather than copied from the old parser's results. Compare candidate recall with packed recall: a drop identifies evidence lost during ranking or packing. The second table shows each method's best candidate rank on a gold page. An absent rank means no annotated page appeared in its top 30.

RRF rewards agreement: a passage ranked 30th in both lists scores `2/90`, greater than `1/61` for a passage ranked first in only one list. This may push useful Dense-only evidence down. These are ten-question development diagnostics, not answer accuracy or a B0/B1/B2 comparison. Page recall does not check whether a selected chunk contains the needed row, year or unit. No retrieval setting is changed automatically.

In [ ]:
display(summary[["mean_candidate_page_recall", "mean_packed_page_recall", "questions_with_all_gold_pages"]].round(3))
gold_ranks = candidate_rows[candidate_rows.on_gold_page].groupby(["financebench_id", "method"])["rank"].min()
best_ranks = gold_ranks.unstack("method").reindex(index=order, columns=METHODS)
display(best_ranks.rename_axis(columns="Best gold-page candidate rank (NaN = absent)"))
print("Corpus:", len(retriever.chunks), "chunks; revision:", chunking_manifest["revision"])
print("Interpretation: page coverage only; missing header/unit context remains a Phase 2 limitation.")

## Step 7 — Inspect misses and save the record

The table below lists only development questions for which Hybrid did not retrieve all annotated pages. Even a retrieved source page may lack the required row, year, unit or negative sign. Conversely, retrieval may find usable evidence outside the annotated pages. This metric therefore does not replace later supported-correctness scoring.

The record saves source hashes, cache revisions, settings, results from the current software test suite, and the execution boundaries: no generator calls and no test evaluation. The data and index have not been rebuilt.

In [ ]:
import subprocess
misses = metrics.query("method == 'Hybrid RRF' and packed_page_recall < 1")
display(misses[["financebench_id", "company", "gold_pages", "candidate_page_recall", "packed_page_recall", "evidence_tokens"]])
miss_details = []
for qid in order:
    for method in METHODS:
        rows = candidate_rows[(candidate_rows.financebench_id == qid) & (candidate_rows.method == method)]
        gold_rows = rows[rows.on_gold_page]
        miss_details.append({"financebench_id": qid, "method": method,
            "best_gold_candidate_rank": int(gold_rows["rank"].min()) if len(gold_rows) else None,
            "gold_candidate_chunks": len(gold_rows), "gold_chunks_selected": int(gold_rows.selected.sum()),
            "diagnosis": "no_gold_page_in_top30" if gold_rows.empty else
                         "some_gold_retained" if gold_rows.selected.any() else "ranking_or_packing_loss"})
pd.DataFrame(miss_details).to_csv(OUT / "development_miss_diagnostics.csv", index=False)
check = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "rag_study", "-p", "test*.py", "-v"],
                       cwd=ROOT, capture_output=True, text=True)
assert check.returncode == 0, check.stdout + check.stderr
(OUT / "software_checks.txt").write_text(check.stdout + check.stderr)
assert all(study.digest(ROOT / "rag_study" / name) == value for name, value in source_hashes.items())
model_files = {}
for label, snapshot in [("embedding", embedding_snapshot), ("generator_tokenizer", tokenizer_snapshot)]:
    model_files[label] = {"revision": snapshot.name, "files_sha256": {f.name: study.digest(f) for f in snapshot.rglob("*") if f.is_file()}}
record = {"checked_at": datetime.now(ZoneInfo("Asia/Hong_Kong")).isoformat(), "config": CONFIG,
          "source_sha256": source_hashes, "cached_model_files": model_files,
          "chunking_revision": chunking_manifest["revision"],
          "parser_sha256": chunking_manifest["parser_sha256"],
          "index_count": len(seen), "index_payload_verified": True, "question_count": len(dev),
          "question_ids": order, "ranking_matches_production": True, "exact_budget_checks_passed": True,
          "summary": summary.reset_index().to_dict(orient="records"), "phase2_document_review_pending_rows": pending,
          "generator_calls": 0, "held_out_evaluation_run": False, "index_rebuilt": False,
          "software_checks_passed": True,
          "software_test_count": sum(line.startswith("test_") for line in check.stderr.splitlines()),
          "scope": "Development retrieval diagnostics only; not B0/B1/B2 answer performance."}
(OUT / "phase3_verification.json").write_text(json.dumps(record, indent=2) + "\n")
print("Phase 3 automated retrieval checks: passed. Model answer generation: not run.")
print("Phase 2 extraction review rows pending:", pending)
print("Review method:", sorted(review.get("review_method", pd.Series(["unrecorded"])).dropna().unique()))
print("AI extraction review is not human answer scoring.")
print("Artifacts:", OUT.relative_to(ROOT))

## Exercises and stopping point

1. In P3_1, choose a question with Hybrid recall = 0 or < 1. Inspect `candidate_rankings.csv`: was its gold page absent from the 30 candidates, or was it dropped during budget packing?
2. Choose an RRF chunk and verify its score using `example_rrf_contributions.csv`. Why cannot this score be called the “probability of a correct answer”?
3. Why might P3_6 show only 1–3 unique pages when four chunks are selected?
4. Why does packed recall = 1 not establish answer correctness? Revisit the Pfizer table example from Phase 2.

**Report additions:** Section 2.3 can describe index consistency, 30 candidates, RRF 60, the four-chunk maximum and complete Qwen token counting. The progress report can include P3_2, explicitly labelled as a ten-question development retrieval diagnostic. Details and values are in `Report/assets/Phase_3_Findings_and_Report_Additions.md`.

**Phase 3 stops here.** Read the six figures, inspect missed-evidence examples and read the completed AI agent audit and its warnings before confirming Phase 4. The separate [Phase 4 notebook](phase_04_generator_and_b0_baseline.ipynb) explains the saved Ollama B0 run. This Phase 3 notebook has not executed Phase 4.